In [18]:
from dotenv import load_dotenv
import os
import pandas as pd
import matplotlib.pyplot as plt
import json


load_dotenv()

RESULTS_FOLDER = os.getenv("RESULTS_FOLDER")
RUNS_FOLDER = os.getenv("RUNS_FOLDER")

In [19]:
results_csv = os.path.join(RESULTS_FOLDER, 'mtsq_test_exact.csv')

df = pd.read_csv(results_csv)

In [20]:
df[["model", "map@5", 'recall@5', 'ndcg@5', 'hit@5',"mrr@5"]]

,model,map@5,recall@5,ndcg@5,hit@5,mrr@5
0,audio_only,0.154972,0.214981,0.170004,0.214981,0.154972
1,text_only,0.000065,0.000086,0.000070,0.000086,0.000065
2,sum,0.033549,0.056197,0.039160,0.056197,0.033549
3,concatenation_42_balanced,0.044147,0.072920,0.051253,0.072920,0.044147


In [21]:
for run in df['run']:
    print(run)

audio_only
text_only
sum
mtsq_concatenation_42_balanced_1028_151724


In [17]:
def plot_losses(train_losses, val_losses, directory, label):
    # Create plot
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label=f'Train {label}', marker='o')
    plt.plot(val_losses, label=f'Val {label}', marker='s')

    # Labels and legend
    plt.xlabel('Epochs')
    plt.ylabel(label)
    plt.title(f'Training and Validation {label}')
    plt.legend()

    # Save the plot
    os.makedirs(directory, exist_ok=True)
    plt.savefig(os.path.join(directory, f"{label}_plot.png"))

    # Close the plot to free memory
    plt.close()

In [25]:
# Run every combination, 3 seeds each.
# Score each config. A config’s score is its mean val map@10 over the 3 seeds.
# Make a table for the shared choice. For each model and each (balance, batch size) cell, take the best score over all the lr and tau values. This is the number for “this cell, once tuned”.
# Choose the cell. Average that best score over the models, and the cell with the highest average wins. When the models disagree, this average settles it. If two cells are within one std of each other, take the cheaper or default one.
# Choose lr and tau per model. With balance and batch size fixed, each model takes the lr and tau with the best seed-mean score.

In [23]:
for run in df[df['trained_on'] == 'mtsq']['run']:
    with open(os.path.join(RUNS_FOLDER, run, "settings.json")) as f:
        saved_settings = json.load(f)

    print(saved_settings)

{'experiment_name': 'mtsq_concatenation_42_balanced_1028_151724', 'timestamp': '1028_151724', 'model_architecture': 'Concatenation(\n  (fc1): Linear(in_features=1024, out_features=512, bias=True)\n)', 'model': 'concatenation', 'dataset': 'mtsq', 'backbone': 'CLAP', 'seed': 42, 'lr': 0.0001, 'tau': 0.05, 'batch_size': 32, 'balance': True, 'epochs': 50, 'train_dataset': '/exp/ldsbrandao/gmr/data/triplets/MTSQ/train_triplets.csv', 'validation_dataset': '/exp/ldsbrandao/gmr/data/triplets/MTSQ/val_triplets.csv', 'n_training_triplets': 371218, 'n_training_songs': 218468, 'n_validation_triplets': 125194, 'n_validation_songs': 73698, 'audio_length_seconds': 10.0, 'n_trainable_parameters': 524800}
